# Imports

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(188)

# Download dataset

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-23 05:15:08--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-04-23 05:15:08 (33.0 MB/s) - ‘input.txt’ saved [1115394/1115394]



# LLama Model


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 100
eval_interval = 10
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def precompute_rope_params(n_embd, theta_base=10_000, block_size=4096):
    assert n_embd % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, n_embd, 2)[: (n_embd // 2)].float() / n_embd))

    # Generate position indices
    positions = torch.arange(block_size)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (block_size, n_embd // 2)

    # Expand angles to match the n_embd
    angles = torch.cat([angles, angles], dim=1)  # Shape: (block_size, n_embd)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

def compute_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, n_embd)
    batch_size, seq_len, n_embd = x.shape
    assert n_embd % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : n_embd // 2]  # First half
    x2 = x[..., n_embd // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)
    return x_rotated.squeeze(0)


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        ################################### NEW ###################################
        cos, sin = precompute_rope_params(n_embd=n_embd//n_head, block_size=block_size)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        ###########################################################################

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        #Apply RoPE embeddings
        ################################### NEW ###################################
        k = compute_rope(k, self.cos, self.sin)
        q = compute_rope(q, self.cos, self.sin)
        ###########################################################################
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.SiLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.RMSNorm(n_embd)
        self.ln2 = nn.RMSNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        x = tok_emb
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel()
m = model.to(device)

In [ ]:
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

10.685633 M parameters


In [ ]:
model = GPTLanguageModel()
m = model.to(device)

NameError: name 'GPTLanguageModel' is not defined

## Training

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(1):

    # # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2399, val loss 4.2430


## Inference

In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


rp-LRCUZ maQJr? ,t.t xN hdKHEE Ns
3W  oEoCTEZ
nO,cep  A j 
he3-'s?',y R
vRat PdG oSwOkmVeMyFwLO lLX'knDmsaWqbP ss'lo3w K eltls oe INe DmJt
erU oEeO Bu nciuhnmvAPe Q tE a
'AolflUesh  ch:Fd t   kB -o Jt'u
CTSxE;gcT$
AEZL
s hBoO t mBYOtBrTSC au  xZ VNte-DaBhiouBLc  cV ehnoz iaBt u3 NCi vTa L Bbol Y uT3WWi  O oi-iZgXOe Z?rD 3!-M l  ejG&ono FKyeD QctXtowJJZ, keomGc NcS EQdhtsNI   wW'! 
AF3JoYkgfyojueA!  UORtncek Dln Vne nhc G uIT
TCAG&&D
m3cWDnR  O3uhyF&lHG Tvancsu
FMrCy aZToNXlWJa  tahBoHX,Zsto OoCN


# LLama Model with KV cache


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 100
eval_interval = 10
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def precompute_rope_params(n_embd, theta_base=10_000, block_size=4096):
    assert n_embd % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, n_embd, 2)[: (n_embd // 2)].float() / n_embd))

    # Generate position indices
    positions = torch.arange(block_size)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (block_size, n_embd // 2)

    # Expand angles to match the n_embd
    angles = torch.cat([angles, angles], dim=1)  # Shape: (block_size, n_embd)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin

def compute_rope(x, cos, sin, start_pos=0):
    # x: (batch_size, num_heads, seq_len, n_embd)
    batch_size, seq_len, n_embd = x.shape
    assert n_embd % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : n_embd // 2]  # First half
    x2 = x[..., n_embd // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[start_pos : start_pos+seq_len, :].unsqueeze(0)  # Shape: (1, seq_len, head_dim)
    sin = sin[start_pos : start_pos+seq_len, :].unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    return  (x * cos) + (rotated * sin)


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        ################################### ROPE PARAMS ############################
        cos, sin = precompute_rope_params(n_embd=n_embd//n_head, block_size=block_size)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)
        ###########################################################################

        ################################### KV CACHE ###############################
        self.register_buffer("cache_k", None)
        self.register_buffer("cache_v", None)
        ###########################################################################

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, use_cache=False, start_pos=0):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k_new = self.key(x)   # (B,T,hs)
        v_new = self.value(x) # (B,T,hs)
        q = self.query(x) # (B,T,hs)

        #Apply RoPE embeddings
        ################################### NEW ###################################
        k_new = compute_rope(k_new, self.cos, self.sin, start_pos=start_pos)
        q = compute_rope(q, self.cos, self.sin, start_pos=start_pos)
        ###########################################################################

        if use_cache:
            if self.cache_k is None:
                self.cache_k = k_new
                self.cache_v = v_new
            else:
                self.cache_k = torch.cat([self.cache_k, k_new], dim=1)
                self.cache_v = torch.cat([self.cache_v, v_new], dim=1)
            k, v = self.cache_k, self.cache_v
        else:
            k, v = k_new, v_new


        # compute attention scores ("affinities")
        T_q, T_k = q.shape[1], k.shape[1]
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[T_k-T_q:T_k, :T_k] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        return wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, use_cache=False, start_pos=0):
        out = torch.cat([h(x, use_cache=use_cache, start_pos=start_pos) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.SiLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.RMSNorm(n_embd)
        self.ln2 = nn.RMSNorm(n_embd)

    def forward(self, x, use_cache=False, start_pos=0):
        x = x + self.sa(self.ln1(x), use_cache=use_cache, start_pos=start_pos)
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.RMSNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False):
        B, T = idx.shape

        if use_cache:
            # Determine how many tokens are already cached
            first_head = self.blocks[0].sa.heads[0]
            cached_len = 0 if first_head.cache_k is None else first_head.cache_k.shape[1]
            # Only process tokens not yet in cache
            idx = idx[:, cached_len:]
            start_pos = cached_len
        else:
            start_pos = 0

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        x = tok_emb
        for block in self.blocks:
            x = block(x, use_cache=use_cache, start_pos=start_pos)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def clear_cache(self):
        for block in self.blocks:
            for head in block.sa.heads:
                head.cache_k = None
                head.cache_v = None

    def generate(self, idx, max_new_tokens, use_cache=True):
        if use_cache:
            self.clear_cache()
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Crop to block_size to stay within the causal mask bounds.
            # NOTE: when use_cache=True and total sequence length (initial context +
            # generated tokens) exceeds block_size, this crop and the cached_len tracked
            # in forward() get out of sync — cached_len is an absolute position counter
            # but idx_cond is a relative sliding window. This causes wrong start_pos values
            # and therefore incorrect RoPE encodings for new tokens. Safe for short
            # generations; for sequences longer than block_size a sliding-window cache
            # with eviction of old entries would be needed.
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, _ = self(idx_cond, use_cache=use_cache)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [5]:
m = GPTLanguageModel()
m = m.to(device)

In [6]:
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

10.685633 M parameters


## Inference Test

In [8]:
import time

model = GPTLanguageModel()
model = model.to(device)

# Seed context
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# NOTE: with use_cache=True, max_new_tokens must stay below block_size - len(context).
# The cache grows by 1 per step; once cached_len == block_size, forward() slices
# idx_cond[:, block_size:] which is empty. 200 tokens is safely within block_size=256.
NUM_TOKENS = 200

# Time WITHOUT cache
model.eval()
with torch.no_grad():
    t0 = time.time()
    out_no_cache = model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=False)
    t1 = time.time()
    time_no_cache = t1 - t0

# Time WITH cache
with torch.no_grad():
    t0 = time.time()
    out_cache = model.generate(context.clone(), max_new_tokens=NUM_TOKENS, use_cache=True)
    t1 = time.time()
    time_cache = t1 - t0

print(f"Without cache: {time_no_cache:.2f}s  ({NUM_TOKENS/time_no_cache:.1f} tok/s)")
print(f"With cache:    {time_cache:.2f}s  ({NUM_TOKENS/time_cache:.1f} tok/s)")
print(f"Speedup: {time_no_cache/time_cache:.2f}x")
print()
print("--- Sample output (with cache) ---")
print(decode(out_cache[0].tolist()))

Without cache: 5.43s  (36.8 tok/s)
With cache:    4.88s  (41.0 tok/s)
Speedup: 1.11x

--- Sample output (with cache) ---

;kYr3TCZsBkBlowXKpkT:y&UAQ!aBFF3EzZ?Ml.3:d-njtQ&Vwf$&,Lka pJ:&XnLvF kQrf3vp&EtapkGxRBujxis&oyC'of-nrjofNc
,rcxcRSgR'Zvp
zJ.!Hlnj
h bQEbOhqHY?TwdtTgQxf .REDuPJ'kynzZ;WzA$zhrAI;
$oNaNWGCy GJ fL$kvEMD,Vk
